In [ ]:
import pandas as pd
from google.colab import files

# Load original AIDev from HuggingFace
print("Loading AIDev from HuggingFace...")
existing = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")
print(f"AIDev PRs: {len(existing):,}")

# Upload your file from PC
print("Upload your file...")
uploaded = files.upload()  # select all_pull_request_extended.parquet from your PC

# Load uploaded file
new_data = pd.read_parquet("all_pull_request_extended.parquet")
print(f"New PRs: {len(new_data):,}")

# Fix agent names
agent_map = {
    "OpenAI Codex": "OpenAI_Codex",
    "GitHub Copilot": "Copilot",
    "Claude Code": "Claude_Code",
}
new_data["agent"] = new_data["agent"].replace(agent_map)

# Fix timestamps
for col in ["created_at", "updated_at", "closed_at", "merged_at", "pull_request_merged_at"]:
    if col in existing.columns:
        existing[col] = pd.to_datetime(existing[col], utc=True, errors="coerce")
    if col in new_data.columns:
        new_data[col] = pd.to_datetime(new_data[col], utc=True, errors="coerce")

# Merge
merged = pd.concat([existing, new_data], ignore_index=True).drop_duplicates(subset=["id"])
print(f"\nFinal merged dataset: {len(merged):,} PRs")

print("\nBreakdown by agent:")
for agent, count in merged["agent"].value_counts().items():
    print(f"  {agent:<22} {count:>10,}")

# Now use merged for your experiments
df = merged
closed_df = df[df['state'] == 'closed']
print(f"\nTotal all PRs:    {df.shape[0]:,}")
print(f"Total closed PRs: {closed_df.shape[0]:,}")
print(f"Closed ratio:     {closed_df.shape[0]/df.shape[0]*100:.1f}%")

Loading AIDev from HuggingFace...
AIDev PRs: 932,791
Upload your file...


Saving all_pull_request_extended.parquet to all_pull_request_extended.parquet
New PRs: 61,532

Final merged dataset: 976,049 PRs

Breakdown by agent:
  OpenAI_Codex              823,748
  Copilot                    58,632
  Cursor                     42,031
  Devin                      38,039
  Claude_Code                13,599

Total all PRs:    976,049
Total closed PRs: 896,806
Closed ratio:     91.9%


In [ ]:
import re

# Keywords related to refactoring
refactor_keywords = r'refactor|refactoring|refactored'
# Search in title and body (description)
mask = (
    closed_df['title'].str.contains(refactor_keywords, case=False, na=False) |
    closed_df['body'].str.contains(refactor_keywords, case=False, na=False)
)

refactor_df = closed_df[mask]

print(f"Refactoring PRs:  {refactor_df.shape[0]:,}")


Refactoring PRs:  36,542


In [ ]:
accepted_refactor = refactor_df[refactor_df['merged_at'].notna()]
rejected_refactor = refactor_df[refactor_df['merged_at'].isna()]

total = len(refactor_df)

print(f"Total refactoring PRs:    {total:,}")
print(f"Accepted (merged):        {len(accepted_refactor):,}  ({len(accepted_refactor)/total*100:.2f}%)")
print(f"Rejected:                 {len(rejected_refactor):,}  ({len(rejected_refactor)/total*100:.2f}%)")

print()

# Total unique repos
total_repos    = refactor_df['repo_full_name'].nunique()
accepted_repos = accepted_refactor['repo_full_name'].nunique()
rejected_repos = rejected_refactor['repo_full_name'].nunique()

print(f"Total refactoring repos:  {total_repos:,}")
print(f"Accepted repos:           {accepted_repos:,}")
print(f"Rejected repos:           {rejected_repos:,}")

Total refactoring PRs:    36,542
Accepted (merged):        32,409  (88.69%)
Rejected:                 4,133  (11.31%)

Total refactoring repos:  2,011
Accepted repos:           1,748
Rejected repos:           353


In [ ]:
import re

# Filter new refactoring PRs only
refactor_keywords = r'refactor|refactoring|refactored'

new_data_refactor = new_data[
    (new_data['state'] == 'closed') &
    (
        new_data['title'].str.contains(refactor_keywords, case=False, na=False) |
        new_data['body'].str.contains(refactor_keywords, case=False, na=False)
    )
]

print(f"New refactoring PRs: {len(new_data_refactor):,}")

# Save just what we need
new_data_refactor[['id', 'html_url', 'agent', 'merged_at']].to_csv("new_refactor_prs.csv", index=False)

from google.colab import files
files.download("new_refactor_prs.csv")

New refactoring PRs: 4,792


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

# Load existing AIDev reviews
print("Loading existing AIDev reviews...")
existing_reviews = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_reviews.parquet")
print(f"Existing reviews: {len(existing_reviews):,}")

# Upload your new reviews file
uploaded = files.upload()  # select pr_reviews_refactor_new.parquet

# Load new reviews
new_reviews = pd.read_parquet("pr_reviews_refactor_new.parquet")
print(f"New reviews: {len(new_reviews):,}")

# Fix timestamps
existing_reviews["submitted_at"] = pd.to_datetime(existing_reviews["submitted_at"], utc=True, errors="coerce")
new_reviews["submitted_at"]      = pd.to_datetime(new_reviews["submitted_at"], utc=True, errors="coerce")

# Merge and deduplicate
merged_reviews = pd.concat([existing_reviews, new_reviews], ignore_index=True).drop_duplicates(subset=["id"])
print(f"\nMerged reviews: {len(merged_reviews):,}")

# Summary
bots   = merged_reviews[merged_reviews["user"].str.contains(r'\[bot\]', case=False, na=False)]
humans = merged_reviews[~merged_reviews["user"].str.contains(r'\[bot\]', case=False, na=False)]
print(f"Human reviews : {len(humans):,}")
print(f"Bot reviews   : {len(bots):,}")

print(f"\nState breakdown:")
print(merged_reviews["state"].value_counts())

# Use in your experiments
reviews_df = merged_reviews

Loading existing AIDev reviews...
Existing reviews: 28,875


Saving pr_reviews_refactor_new.parquet to pr_reviews_refactor_new.parquet
New reviews: 5,804

Merged reviews: 34,425
Human reviews : 18,825
Bot reviews   : 15,600

State breakdown:
state
COMMENTED            25205
APPROVED              7065
CHANGES_REQUESTED     1793
DISMISSED              362
Name: count, dtype: int64


In [ ]:
# Step 1: Filter out bots - bots have [bot] in their username
human_reviews = reviews_df[~reviews_df['user'].str.contains(r'\[bot\]', case=False, na=False)]

# Step 2: Count human reviews per PR
human_review_counts = human_reviews.groupby('pr_id')['id'].count().reset_index()
human_review_counts.columns = ['id', 'human_review_count']

# Step 3: Keep only PRs with at least 1 human review
prs_with_human_review = human_review_counts[human_review_counts['human_review_count'] >= 1]

# Step 4: Extract accepted refactoring PRs with at least 1 human review
accepted_with_reviews = accepted_refactor[accepted_refactor['id'].isin(prs_with_human_review['id'])]

# Step 5: Extract rejected refactoring PRs with at least 1 human review
rejected_with_reviews = rejected_refactor[rejected_refactor['id'].isin(prs_with_human_review['id'])]

# Step 6: Print summary
print(f"Accepted refactoring PRs with at least 1 human review: {len(accepted_with_reviews):,}")
print(f"Rejected refactoring PRs with at least 1 human review: {len(rejected_with_reviews):,}")

# Step 7: Total unique repos
accepted_repos = len(accepted_with_reviews['repo_full_name'].unique())
rejected_repos = len(rejected_with_reviews['repo_full_name'].unique())
total_repos    = len(pd.concat([accepted_with_reviews, rejected_with_reviews])['repo_full_name'].unique())

print(f"\nTotal repos (accepted + rejected): {total_repos:,}")
print(f"Accepted repos                   : {accepted_repos:,}")
print(f"Rejected repos                   : {rejected_repos:,}")

# Step 8: Download
from google.colab import files

accepted_with_reviews.to_csv("accepted_with_reviews_mix.csv", index=False)
rejected_with_reviews.to_csv("rejected_with_reviews_mix.csv", index=False)

files.download("accepted_with_reviews_mix.csv")
files.download("rejected_with_reviews_mix.csv")



Accepted refactoring PRs with at least 1 human review: 842
Rejected refactoring PRs with at least 1 human review: 95

Total repos (accepted + rejected): 347
Accepted repos                   : 320
Rejected repos                   : 34


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from openai import OpenAI
import pandas as pd
import time
from google.colab import files

# Upload your files
uploaded = files.upload()  # upload both CSV files

client = OpenAI(api_key="api_key")

# Load files
accepted_with_reviews = pd.read_csv("accepted_with_reviews_mix.csv")
rejected_with_reviews = pd.read_csv("rejected_with_reviews_mix.csv")

# Add labels
accepted_with_reviews['label'] = 'accepted'
rejected_with_reviews['label'] = 'rejected'

all_prs = pd.concat([accepted_with_reviews, rejected_with_reviews]).reset_index(drop=True)
total = len(all_prs)
print(f"Total PRs to classify: {total:,}")
print(f"Accepted: {len(accepted_with_reviews):,} | Rejected: {len(rejected_with_reviews):,}")

# Step 1 — classify into main category first
def classify_category(title, body):
    prompt = f"""You are a software engineering expert. Classify this refactoring pull request into ONE of these categories:
- Internal Quality Attribute
- External Quality Attribute
- Code Smell

Title: {title}
Description: {str(body)[:500]}

Reply with ONLY the category name."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

# Step 2 — classify into specific type
def classify_specific(title, body, category):
    if category == "Internal Quality Attribute":
        options = "Dependency, Inheritance, Composition, Abstraction, Coupling, Encapsulation, Polymorphism, Complexity"
    elif category == "External Quality Attribute":
        options = "Readability, Usability, Performance, Maintainability, Flexibility, Reusability, Accessibility, Modularity, Extensibility, Correctness, Robustness, Scalability, Simplicity, Reliability, Understandability"
    else:
        options = "Duplicate Code, Long Method, Dead Code, Bad Naming, Large Class, Feature Envy, Generic Code Smell"

    prompt = f"""You are a software engineering expert. Given this pull request belongs to the category '{category}', identify the most specific type from this list:

{options}

Title: {title}
Description: {str(body)[:500]}

Reply with ONLY the exact type name from the list above."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=15,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

# Run classification
categories    = []
specific_types = []

for i, row in all_prs.iterrows():
    try:
        # Step 1: get category
        category = classify_category(row['title'], row['body'])
        categories.append(category)

        # Step 2: get specific type
        specific = classify_specific(row['title'], row['body'], category)
        specific_types.append(specific)

        print(f"[{i+1}/{total}] {row['agent']} | {row['label']} | {category} → {specific}")
        time.sleep(0.3)

    except Exception as e:
        categories.append("Unknown")
        specific_types.append("Unknown")
        print(f"[{i+1}/{total}] Error: {e}")

all_prs['category']      = categories
all_prs['specific_type'] = specific_types

print(f"\nDone! Classified {total:,} PRs")
print("\nCategory breakdown:")
print(all_prs['category'].value_counts())
print("\nTop specific types:")
print(all_prs['specific_type'].value_counts().head(10))
# Save and download
all_prs.to_csv("classified_refactor_prs_mix.csv", index=False)
files.download("classified_refactor_prs_mix.csv")
print("Downloaded: classified_refactor_prs_mix.csv")

Saving accepted_with_reviews_mix.csv to accepted_with_reviews_mix (2).csv
Saving rejected_with_reviews_mix.csv to rejected_with_reviews_mix (2).csv
Total PRs to classify: 937
Accepted: 842 | Rejected: 95
[1/937] Claude_Code | accepted | Internal Quality Attribute → Complexity
[2/937] Claude_Code | accepted | Internal Quality Attribute → Composition
[3/937] Claude_Code | accepted | Internal Quality Attribute → Abstraction
[4/937] Claude_Code | accepted | Internal Quality Attribute → Encapsulation
[5/937] Claude_Code | accepted | Internal Quality Attribute → Coupling
[6/937] Claude_Code | accepted | Internal Quality Attribute → Complexity
[7/937] Claude_Code | accepted | Internal Quality Attribute → Dependency
[8/937] Claude_Code | accepted | Internal Quality Attribute → Coupling
[9/937] Claude_Code | accepted | Internal Quality Attribute → Coupling
[10/937] Claude_Code | accepted | Internal Quality Attribute → Complexity
[11/937] Claude_Code | accepted | Internal Quality Attribute → Com

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: classified_refactor_prs_mix.csv


In [ ]:
df = pd.read_csv("classified_refactor_prs_mix.csv")

total_all = len(df)
print(f"Total classified PRs: {total_all:,}")
print(f"Accepted: {len(df[df['label']=='accepted']):,}")
print(f"Rejected: {len(df[df['label']=='rejected']):,}")

print(f"\n{'='*75}")

categories = {
    "Internal Quality Attribute": [
        "Dependency", "Inheritance", "Composition", "Abstraction",
        "Coupling", "Encapsulation", "Polymorphism", "Complexity"
    ],
    "External Quality Attribute": [
        "Readability", "Usability", "Performance", "Maintainability",
        "Flexibility", "Reusability", "Accessibility", "Modularity",
        "Extensibility", "Correctness", "Robustness", "Scalability",
        "Simplicity", "Reliability", "Understandability"
    ],
    "Code Smell": [
        "Duplicate Code", "Long Method", "Dead Code", "Bad Naming",
        "Large Class", "Feature Envy", "Generic Code Smell"
    ]
}

rows = []

for category, types in categories.items():
    cat_df    = df[df['category'] == category]
    cat_total = len(cat_df)
    cat_pct   = cat_total / total_all * 100

    print(f"\n{category} — Total: {cat_total:,} ({cat_pct:.1f}%)")
    print(f"  {'Type':<20} {'Total':>6} {'%/937':>7} {'Accepted':>9} {'%/937':>7} {'Rejected':>9} {'%/937':>7}")
    print(f"  {'─'*73}")

    for t in types:
        type_df  = cat_df[cat_df['specific_type'] == t]
        accepted = len(type_df[type_df['label'] == 'accepted'])
        rejected = len(type_df[type_df['label'] == 'rejected'])
        total    = len(type_df)
        if total > 0:
            pct_total = total    / total_all * 100
            pct_acc   = accepted / total_all * 100
            pct_rej   = rejected / total_all * 100
            print(f"  {t:<20} {total:>6,} {pct_total:>6.1f}% {accepted:>9,} {pct_acc:>6.1f}% {rejected:>9,} {pct_rej:>6.1f}%")
            rows.append({
                "category":          category,
                "specific_type":     t,
                "total":             total,
                "pct_of_937":        round(pct_total, 1),
                "accepted":          accepted,
                "pct_accepted_937":  round(pct_acc, 1),
                "rejected":          rejected,
                "pct_rejected_937":  round(pct_rej, 1),
            })

print(f"\n{'='*75}")
print("SUMMARY BY CATEGORY")
print(f"{'='*75}")
print(f"  {'Category':<30} {'Total':>6} {'%/937':>7} {'Accepted':>9} {'%/937':>7} {'Rejected':>9} {'%/937':>7}")
print(f"  {'─'*79}")

for category in categories:
    cat_df   = df[df['category'] == category]
    accepted = len(cat_df[cat_df['label'] == 'accepted'])
    rejected = len(cat_df[cat_df['label'] == 'rejected'])
    total    = len(cat_df)
    if total > 0:
        pct_total = total    / total_all * 100
        pct_acc   = accepted / total_all * 100
        pct_rej   = rejected / total_all * 100
        print(f"  {category:<30} {total:>6,} {pct_total:>6.1f}% {accepted:>9,} {pct_acc:>6.1f}% {rejected:>9,} {pct_rej:>6.1f}%")

Total classified PRs: 937
Accepted: 842
Rejected: 95


Internal Quality Attribute — Total: 779 (83.1%)
  Type                  Total   %/937  Accepted   %/937  Rejected   %/937
  ─────────────────────────────────────────────────────────────────────────
  Dependency               45    4.8%        38    4.1%         7    0.7%
  Inheritance               4    0.4%         3    0.3%         1    0.1%
  Composition              39    4.2%        37    3.9%         2    0.2%
  Abstraction             433   46.2%       387   41.3%        46    4.9%
  Coupling                 68    7.3%        61    6.5%         7    0.7%
  Encapsulation            52    5.5%        47    5.0%         5    0.5%
  Complexity              135   14.4%       126   13.4%         9    1.0%

External Quality Attribute — Total: 128 (13.7%)
  Type                  Total   %/937  Accepted   %/937  Rejected   %/937
  ─────────────────────────────────────────────────────────────────────────
  Readability               4 

In [ ]:
import pandas as pd
from scipy import stats
from google.colab import files

# Load files
accepted = pd.read_csv("accepted_with_reviews_mix.csv")
rejected = pd.read_csv("rejected_with_reviews_mix.csv")

accepted['label'] = 'accepted'
rejected['label'] = 'rejected'

df = pd.concat([accepted, rejected], ignore_index=True)

# Load classified file to get category/specific_type
classified = pd.read_csv("classified_refactor_prs_mix.csv")
df = df.merge(classified[['id', 'category', 'specific_type', 'label']],
              on='id', how='left', suffixes=('', '_cls'))

# Calculate turnaround time in hours
df['created_at'] = pd.to_datetime(df['created_at'], utc=True, errors='coerce')
df['closed_at']  = pd.to_datetime(df['closed_at'],  utc=True, errors='coerce')
df['turnaround_hours'] = (df['closed_at'] - df['created_at']).dt.total_seconds() / 3600

print(f"Total PRs: {len(df):,}")
print(f"Accepted: {len(df[df['label']=='accepted']):,}")
print(f"Rejected: {len(df[df['label']=='rejected']):,}")
print(f"\nMean turnaround (accepted): {df[df['label']=='accepted']['turnaround_hours'].mean():.1f} hours")
print(f"Mean turnaround (rejected): {df[df['label']=='rejected']['turnaround_hours'].mean():.1f} hours")

# Statistical test per specific type
internal_df = df[df['category'] == 'Internal Quality Attribute']

specific_types = ['Abstraction', 'Dependency', 'Coupling', 'Complexity',
                  'Composition', 'Encapsulation', 'Inheritance']

def cliffs_delta(acc, rej):
    n1, n2 = len(acc), len(rej)
    if n1 == 0 or n2 == 0: return None
    greater = sum(1 for a in acc for r in rej if a > r)
    less    = sum(1 for a in acc for r in rej if a < r)
    return abs((greater - less) / (n1 * n2))

def cliff_label(d):
    if d is None: return "N/A"
    if d < 0.147: return f"{d:.3f} (N)"
    elif d < 0.33: return f"{d:.3f} (S)"
    elif d < 0.474: return f"{d:.3f} (M)"
    else: return f"{d:.3f} (L)"

print(f"\n{'Specific Type':<15} {'p-value':<20} {'Significant':<15} {'Cliff Delta'}")
print("─" * 65)

for t in specific_types:
    type_df  = internal_df[internal_df['specific_type'] == t]
    acc_vals = type_df[type_df['label'] == 'accepted']['turnaround_hours'].dropna().tolist()
    rej_vals = type_df[type_df['label'] == 'rejected']['turnaround_hours'].dropna().tolist()

    if len(acc_vals) == 0 or len(rej_vals) == 0:
        print(f"{t:<15} {'N/A':<20} {'N/A':<15} N/A")
        continue

    _, pval = stats.mannwhitneyu(acc_vals, rej_vals, alternative='two-sided')
    sig     = "Yes" if pval < 0.05 else "No"
    cd      = cliffs_delta(acc_vals, rej_vals)
    print(f"{t:<15} {pval:<20.3e} {sig:<15} {cliff_label(cd)}")

Total PRs: 937
Accepted: 842
Rejected: 95

Mean turnaround (accepted): 70.4 hours
Mean turnaround (rejected): 347.4 hours

Specific Type   p-value              Significant     Cliff Delta
─────────────────────────────────────────────────────────────────
Abstraction     4.299e-15            Yes             0.707 (L)
Dependency      1.139e-03            Yes             0.737 (L)
Coupling        1.737e-01            No              0.321 (S)
Complexity      6.947e-04            Yes             0.679 (L)
Composition     3.266e-01            No              0.459 (M)
Encapsulation   1.314e-01            No              0.421 (M)
Inheritance     5.000e-01            No              1.000 (L)


In [ ]:
import pandas as pd
from scipy import stats

# Load merged classified file
df = pd.read_csv("classified_refactor_prs_mix.csv")

# Filter Code Smell only
codesmell_df = df[df['category'] == 'Code Smell']

specific_types = ['Dead Code', 'Duplicate Code', 'Long Method',
                  'Bad Naming', 'Generic Code Smell']

def cliffs_delta(acc, rej):
    n1, n2 = len(acc), len(rej)
    if n1 == 0 or n2 == 0: return None
    greater = sum(1 for a in acc for r in rej if a > r)
    less    = sum(1 for a in acc for r in rej if a < r)
    return abs((greater - less) / (n1 * n2))

def cliff_label(d):
    if d is None: return "N/A"
    if d < 0.147: return f"{d:.3f} (N)"
    elif d < 0.33: return f"{d:.3f} (S)"
    elif d < 0.474: return f"{d:.3f} (M)"
    else: return f"{d:.3f} (L)"

print(f"{'Specific Type':<20} {'p-value':<20} {'Significant':<15} {'Cliff Delta'}")
print("─" * 70)

for t in specific_types:
    type_df  = codesmell_df[codesmell_df['specific_type'] == t]
    accepted = type_df[type_df['label'] == 'accepted']['id'].tolist()
    rejected = type_df[type_df['label'] == 'rejected']['id'].tolist()

    if len(accepted) == 0 or len(rejected) == 0:
        print(f"{t:<20} {'N/A':<20} {'N/A':<15} N/A")
        continue

    _, pval = stats.mannwhitneyu(accepted, rejected, alternative='two-sided')
    sig     = "Yes" if pval < 0.05 else "No"
    cd      = cliffs_delta(accepted, rejected)
    print(f"{t:<20} {pval:<20.3e} {sig:<15} {cliff_label(cd)}")

Specific Type        p-value              Significant     Cliff Delta
──────────────────────────────────────────────────────────────────────
Dead Code            2.735e-02            Yes             0.733 (L)
Duplicate Code       N/A                  N/A             N/A
Long Method          N/A                  N/A             N/A
Bad Naming           N/A                  N/A             N/A
Generic Code Smell   6.667e-01            No              1.000 (L)


In [ ]:
import pandas as pd
from scipy import stats

# Load merged classified file
df = pd.read_csv("classified_refactor_prs_mix.csv")

# Filter External Quality Attribute only
external_df = df[df['category'] == 'External Quality Attribute']

specific_types = ['Readability', 'Usability', 'Performance', 'Maintainability',
                  'Flexibility', 'Reusability', 'Accessibility', 'Modularity',
                  'Extensibility', 'Correctness', 'Robustness', 'Scalability',
                  'Simplicity', 'Reliability', 'Understandability']

def cliffs_delta(acc, rej):
    n1, n2 = len(acc), len(rej)
    if n1 == 0 or n2 == 0: return None
    greater = sum(1 for a in acc for r in rej if a > r)
    less    = sum(1 for a in acc for r in rej if a < r)
    return abs((greater - less) / (n1 * n2))

def cliff_label(d):
    if d is None: return "N/A"
    if d < 0.147: return f"{d:.3f} (N)"
    elif d < 0.33: return f"{d:.3f} (S)"
    elif d < 0.474: return f"{d:.3f} (M)"
    else: return f"{d:.3f} (L)"

print(f"{'Specific Type':<20} {'p-value':<20} {'Significant':<15} {'Cliff Delta'}")
print("─" * 70)

for t in specific_types:
    type_df  = external_df[external_df['specific_type'] == t]
    accepted = type_df[type_df['label'] == 'accepted']['id'].tolist()
    rejected = type_df[type_df['label'] == 'rejected']['id'].tolist()

    if len(accepted) == 0 or len(rejected) == 0:
        print(f"{t:<20} {'N/A':<20} {'N/A':<15} N/A")
        continue

    _, pval = stats.mannwhitneyu(accepted, rejected, alternative='two-sided')
    sig     = "Yes" if pval < 0.05 else "No"
    cd      = cliffs_delta(accepted, rejected)
    print(f"{t:<20} {pval:<20.3e} {sig:<15} {cliff_label(cd)}")

Specific Type        p-value              Significant     Cliff Delta
──────────────────────────────────────────────────────────────────────
Readability          N/A                  N/A             N/A
Usability            8.551e-01            No              0.061 (N)
Performance          1.000e+00            No              0.000 (N)
Maintainability      6.000e-01            No              0.556 (L)
Flexibility          1.000e+00            No              0.000 (N)
Reusability          N/A                  N/A             N/A
Accessibility        N/A                  N/A             N/A
Modularity           N/A                  N/A             N/A
Extensibility        N/A                  N/A             N/A
Correctness          2.448e-01            No              0.385 (M)
Robustness           1.000e+00            No              1.000 (L)
Scalability          N/A                  N/A             N/A
Simplicity           N/A                  N/A             N/A
Reliability      